In [1]:
import sys
import os
import requests

sys.path.append(os.path.abspath(".."))

In [2]:
from supabase import create_client
import os
import pandas as pd
from dotenv import load_dotenv
from config import (
    CLEAN_DATA_NAME, # "clean_data"
    FEATURE_STORE_NAME, # "feature_store"
    FEEDBACK_STORE_NAME # "feedback_store"
)

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_PUBLIC_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

### This table is the etl data freshly extracted from a google sheet for a late predictor project

In [3]:
clean_data_res = (
    supabase
    .table(CLEAN_DATA_NAME)
    .select("*")
    .execute()
)

clean_data_df = pd.DataFrame(clean_data_res.data)

clean_data_df

,id,date,meeting_time,arrived_time,meeting_location,category
0,766145a1-54e5-48d7-9436-fb0aff2b97cc,2026-01-02,2026-01-02T03:00:00+00:00,2026-01-02T03:07:00+00:00,uncle kkm (bukit panjang),lunch
1,07c7e122-1141-46c8-8995-680148a6e1eb,2026-01-03,2026-01-03T01:15:00+00:00,2026-01-03T01:20:00+00:00,virtual,apply job
2,3038a55f-a370-47f0-9a7f-10f7fa0f9a58,2026-01-06,2026-01-06T11:30:00+00:00,2026-01-06T12:05:00+00:00,bbdc,exercise
3,d4b2df08-0531-48f7-9bcd-f3e7076a6bf0,2026-01-08,2026-01-08T11:00:00+00:00,2026-01-08T11:48:00+00:00,raffles place,dinner/drinks
4,27434e7a-3a12-40ee-a6aa-83874056218a,2026-01-15,2026-01-15T11:00:00+00:00,NaN,imm,dinner/drinks
5,5db33145-7a80-4799-91dd-4a9d0576d012,2026-01-23,2026-01-23T12:00:00+00:00,2026-01-23T13:00:00+00:00,cck,dinner/drinks
6,2285f8d1-b588-4c98-b453-a6a0cc2958ee,2026-03-08,2026-03-08T01:20:00+00:00,2026-03-08T01:30:00+00:00,uncle kkm (bukit panjang),breakfast
7,211dbc12-deb5-474a-b72f-7bf0de6a1e15,2026-03-27,2026-03-27T12:00:00+00:00,2026-03-27T12:15:00+00:00,her house,dinner/drinks
8,d90f0d0c-82b0-4b3f-a4fa-77ac2ffce977,2026-03-28,2026-03-28T07:00:00+00:00,2026-03-28T07:30:00+00:00,suntec,work/career fair
9,57af8e4f-6f91-4b3a-aca9-9bed44fac25f,2026-04-10,2026-04-10T10:30:00+00:00,2026-04-10T10:39:00+00:00,suntec,dinner/drinks


### This table contains the feature store after processing the cleaned data

In [4]:
feature_store_res = (
    supabase
    .table(FEATURE_STORE_NAME)
    .select("*")
    .execute()
)

feature_store_df = pd.DataFrame(feature_store_res.data)

feature_store_df

,id,day_of_week,distance_km,category,late_duration_min,time_of_day,clean_data_id
0,94d0b7e4-ba9d-4d3c-900e-660a4402e051,4,1.274747,lunch,7,morning,766145a1-54e5-48d7-9436-fb0aff2b97cc
1,f10dd5df-d779-47e0-911e-89b95dee85d1,1,1.773078,exercise,35,evening,3038a55f-a370-47f0-9a7f-10f7fa0f9a58
2,5723363d-59af-41be-8f73-2e90cbe0438e,3,15.205063,dinner/drinks,48,evening,d4b2df08-0531-48f7-9bcd-f3e7076a6bf0
3,62d7579d-4ea3-4173-888d-e9a5d4ec87fd,4,1.130494,dinner/drinks,60,evening,5db33145-7a80-4799-91dd-4a9d0576d012
4,eebc6f9a-e703-4683-a7b6-f964fcccbaba,6,1.274747,breakfast,10,morning,2285f8d1-b588-4c98-b453-a6a0cc2958ee
5,d9d610cd-cb4f-43d3-afae-2d2b1d7a227b,5,15.077114,work/career fair,30,afternoon,d90f0d0c-82b0-4b3f-a4fa-77ac2ffce977
6,fa9488ab-026e-48aa-a2ee-f0abae24c51a,4,15.077114,dinner/drinks,9,evening,57af8e4f-6f91-4b3a-aca9-9bed44fac25f
7,b1cee7dc-3b60-461e-b2bb-823c8682c809,4,1.274747,lunch,13,afternoon,f407f8f5-f81b-4e7d-bba7-c93abc0c8c05
8,37a7442f-41eb-43f2-84e8-83f28278241c,1,5.656470,lunch,17,morning,557800bd-1769-4915-b4f3-9086bc2b2cea
9,43373a44-c6a1-4e6e-9c42-7cce7ab067fa,2,0.871136,dinner/drinks,21,evening,984df188-1d9a-4dc2-ada4-3023e7f56f50


### This table contains the feedback data provided by users after predictions has been made

In [5]:
feedback_store_res = (
    supabase
    .table(FEEDBACK_STORE_NAME)
    .select("*")
    .execute()
)

feedback_store_df = pd.DataFrame(feedback_store_res.data)

feedback_store_df

,id,created_at,day_of_week,distance_km,time_of_day,late_duration_min,pred_late_duration_min,category,models_used,feature_registry_id
0,6bd42249-7e33-4d9e-bae7-83e3ec126203,2026-05-10T10:37:17.738969+00:00,2,12.43,afternoon,20,19.004129,dinner,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
1,7e8590d8-9a5d-4609-b26c-b4abf278de92,2026-05-10T10:40:13.891755+00:00,6,14.48,evening,-480,24.000000,exercise,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
2,403608cd-1550-454c-a562-b20105a33830,2026-05-10T10:49:10.59637+00:00,6,14.60,evening,61,24.000000,apply job,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
3,ff1a6d47-2644-4b72-8e8d-8edb86568da2,2026-05-10T10:52:32.643209+00:00,6,14.48,evening,1320,24.000000,apply job,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
4,2c71c04a-30f5-4b84-89d1-4d160d860dfb,2026-05-10T10:52:55.944985+00:00,0,9.34,evening,-12602,32.000000,dinner/drinks,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
5,c31fef06-28a4-4d93-8d09-fd394325909e,2026-05-10T10:55:14.812585+00:00,6,14.41,evening,60,24.000000,apply job,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
6,c3d9ba00-4d90-4a2e-8b2f-deb3567058f9,2026-05-10T11:04:22.161828+00:00,3,16.04,evening,15840,34.000000,apply job,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
7,09958433-e758-4a79-98d2-453402e46593,2026-05-10T11:10:33.408651+00:00,6,27.69,evening,1470,24.000000,work/career fair,"random_forest, ridge",8fd7d397-e3b6-4289-b5b0-3af28f89092f
